# Vision Transformer (ViT)

**Domain:** Architectures  ·  **recommended addition**  ·  **runnable:** yes

A refresher on the **Vision Transformer** — the architecture that showed you can drop convolutions
entirely and classify images with a plain Transformer encoder, by chopping the image into patches and
treating each patch as a token (Dosovitskiy et al., *An Image is Worth 16×16 Words*, 2020).

## 1. What & Why

A **Vision Transformer** applies the standard Transformer *encoder* — the exact same block used in BERT —
to images. The only new idea is how to turn a 2-D image into a sequence of tokens: **cut it into a grid of
fixed-size patches** (typically 16×16 pixels), flatten each patch, and linearly project it to a token
embedding. A 224×224 image at patch size 16 becomes 14×14 = **196 tokens**, plus one learnable **[CLS]**
token whose final state is fed to the classifier. Add positional embeddings, run the stack of self-attention
+ MLP blocks, read off the CLS token. That's it — no convolutions anywhere.

**The problem it solves.** CNNs bake in strong priors — **locality** (pixels near each other interact first)
and **translation equivariance** (a feature detector slides across the image). Those priors are great for
small/medium data but they *cap* how the model relates distant parts of an image: long-range interactions
only emerge deep in the network after many pooling steps. Self-attention is **global from layer one** — every
patch can attend to every other patch immediately — and it scales: with enough data, a ViT learns better
visual representations than a same-budget CNN, and keeps improving as you add data and parameters.

**The catch.** Those CNN priors are also a *free* form of regularization. Without them, ViTs are **data
hungry**: trained from scratch on ImageNet-1k (1.3M images) a ViT *underperforms* a ResNet. The magic only
appears with large-scale pretraining (ImageNet-21k, JFT-300M) or strong augmentation/distillation recipes
(DeiT).

**Reach for it when:** you have lots of data or a good pretrained ViT, you need global context (scene
understanding, fine-grained relations), or you want one architecture shared across modalities (the same
encoder powers CLIP, multimodal LLMs, segmentation, detection). **Don't** reach for it when data is scarce
and you can't pretrain, latency/compute is tight at high resolution, or a well-tuned CNN already nails the
task — the convolutional prior is hard to beat in the small-data regime.

## 2. Mental Model

**An image is a sentence; each 16×16 patch is a word.**

ViT does to a picture exactly what BERT does to text — it just needs a "tokenizer" for pixels:

```
   image 224×224×3                 sequence of 197 tokens          Transformer encoder
 ┌───┬───┬───┬───┐               [CLS] p1 p2 p3 ... p196          ┌──────────────────┐
 │ p1│ p2│ p3│...│   patchify    each patch → flatten → Linear     │  self-attention   │
 ├───┼───┼───┼───┤  ─────────▶   → D-dim token, + position  ─────▶ │  + MLP, ×L layers │ ─▶ [CLS] → MLP head → class
 │   │   │   │   │               (CLS is a learnable token)        └──────────────────┘
 └───┴───┴───┴───┘
   14×14 = 196 patches
```

Three things make it click:

- **The patch embedding is just a strided convolution.** Projecting each non-overlapping 16×16 patch to a
  D-dim vector is identical to a `Conv2d(in, D, kernel=16, stride=16)`. So even "no convolutions" sneaks one
  in at the very front — but only one, with no overlap.
- **Position must be *told*, not assumed.** Attention is permutation-invariant: shuffle the patches and a raw
  Transformer can't tell. The learnable **positional embeddings** are what reintroduce "where each patch was."
- **The [CLS] token is a sponge.** It carries no patch content; through the layers it attends to all patches
  and soaks up a global summary, which the classification head reads. (Alternatives: average-pool all patch
  tokens instead — works about as well.)

## 3. Key Concepts

- **Patch embedding** — split the image into a grid of P×P patches, flatten, linearly project to dimension D.
  Equivalent to a `Conv2d` with kernel = stride = P. Patch size P trades resolution for cost (smaller P → more
  tokens → more compute).
- **[CLS] token** — a single learnable embedding prepended to the patch sequence; its output is the image
  representation used for classification. Global average pooling over patch tokens is a common alternative.
- **Positional embedding** — learnable per-position vectors added to tokens so the model knows patch layout.
  Because they're tied to a fixed grid, **changing input resolution requires interpolating** them.
- **Self-attention (global mixing)** — every token attends to every token; cost is **O(N²)** in the number of
  patches N. This is the source of both ViT's global receptive field and its compute bill.
- **Encoder block** — the standard pre-norm Transformer: `x += Attn(LN(x)); x += MLP(LN(x))`, MLP usually 4×
  the width with GELU. Stacked L times (ViT-B = 12 layers, D=768, 12 heads, ~86M params).
- **Inductive bias / data hunger** — ViT lacks the CNN's locality + translation-equivariance priors, so it
  needs large-scale pretraining or heavy augmentation to match CNNs; once it has that data, it surpasses them.
- **Variants worth knowing** — **DeiT** (data-efficient training + distillation token, no JFT needed),
  **Swin** (windowed/shifted attention → linear cost, hierarchical like a CNN), **ViT-MAE** (masked-patch
  self-supervised pretraining), **DINO/DINOv2** (self-supervised features). ViT is also the image encoder in
  **CLIP** and most multimodal LLMs.

## 4. Setup

The worked examples use only **PyTorch (CPU is fine)** plus NumPy. We build the patch embedding and a tiny
ViT from scratch with `torch.nn`, so the mechanics are visible — no GPU, no downloads, no API keys.

The last cell shows how to load a *real* pretrained `vit_b_16` from torchvision; it's **gated** behind an
`os.getenv("ALLOW_DOWNLOAD")` check (it would download weights), so the notebook runs offline either way.

In [1]:
# %pip install numpy torch torchvision   # CPU build of torch is enough
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)
print("numpy", np.__version__, "| torch", torch.__version__)

numpy 2.4.6 | torch 2.12.1


## 5. Worked Examples

### Example 1 — Patchify: turn an image into a sequence of tokens

The whole "image → tokens" step. We take a 32×32×3 image, project non-overlapping 16×16 patches to D-dim
tokens with a strided `Conv2d` (the canonical ViT patch embed), then prepend a **[CLS]** token and add
**positional** embeddings. Watch the shapes: a 4-D image becomes a `(batch, n_tokens, D)` sequence — exactly
what a text Transformer eats.

In [2]:
C, H, W = 3, 32, 32      # channels, height, width
P = 16                   # patch size  -> a 2x2 grid = 4 patches
D = 64                   # embedding dim (token width)

img = torch.randn(1, C, H, W)                       # (batch=1, C, H, W)

# Patch embed: a strided conv == linear projection of each non-overlapping patch.
patch_embed = nn.Conv2d(C, D, kernel_size=P, stride=P)
feat = patch_embed(img)                             # (1, D, H/P, W/P)
n_h, n_w = feat.shape[-2:]
tokens = feat.flatten(2).transpose(1, 2)            # (1, n_patches, D)
n_patches = tokens.shape[1]
print(f"image {tuple(img.shape)} -> grid {n_h}x{n_w} -> {n_patches} patch tokens of dim {D}: {tuple(tokens.shape)}")

# Prepend a learnable [CLS] token and add (learnable) positional embeddings.
cls = nn.Parameter(torch.zeros(1, 1, D))
pos = nn.Parameter(torch.randn(1, n_patches + 1, D) * 0.02)
seq = torch.cat([cls.expand(1, -1, -1), tokens], dim=1) + pos
print("sequence into the Transformer:", tuple(seq.shape), " (+1 token for [CLS])")

image (1, 3, 32, 32) -> grid 2x2 -> 4 patch tokens of dim 64: (1, 4, 64)
sequence into the Transformer: (1, 5, 64)  (+1 token for [CLS])


### Example 2 — A tiny ViT, end to end

Now assemble the full path: patch embed → prepend CLS + add positions → a stack of standard Transformer
encoder layers (`nn.TransformerEncoderLayer`) → classify from the CLS token. This is a real, if small, ViT
that runs a forward pass on CPU. Note the parameter count — even this toy has the familiar Transformer bulk.

In [3]:
class TinyViT(nn.Module):
    def __init__(self, img_size=32, patch=16, in_ch=3, dim=64, depth=4, heads=4, n_classes=10):
        super().__init__()
        n_patches = (img_size // patch) ** 2
        self.patch_embed = nn.Conv2d(in_ch, dim, patch, patch)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, n_patches + 1, dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            dim, heads, dim * 4, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, depth)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)   # (B, n_patches, dim)
        cls = self.cls.expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos
        x = self.encoder(x)
        return self.head(self.norm(x[:, 0]))                 # classify from the CLS token

model = TinyViT()
logits = model(torch.randn(2, 3, 32, 32))
n_params = sum(p.numel() for p in model.parameters())
print("logits:", tuple(logits.shape), "| params:", f"{n_params:,}")
print("predicted classes:", logits.argmax(-1).tolist())

logits: (2, 10) | params: 250,314
predicted classes: [4, 4]


/var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/ipykernel_55644/3018751438.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, depth)


### Example 3 — Why patch size dominates the compute bill

Self-attention is **O(N²)** in the number of patch tokens N, and N = (image_size / patch_size)². So the patch
size is the single biggest cost lever: halving it quadruples N and **~16×** the attention cost. This is why
ViTs settled on 16×16 patches, and why high-resolution ViTs get expensive fast (and why Swin switched to
windowed attention).

In [4]:
print(f"{'image':>9} | {'patch':>6} | {'tokens N':>9} | {'attn ~ N^2':>14}")
print("-" * 48)
for img_sz, patch in [(224, 32), (224, 16), (224, 8), (384, 16), (512, 16)]:
    n = (img_sz // patch) ** 2
    print(f"{img_sz:>6}px | {patch:>4}px | {n:>9,} | {n * n:>14,}")
print("\nHalving the patch size 4x's the token count and ~16x's the attention cost.")
print("Bigger images do the same -> high-res ViTs are costly; this is why Swin uses windowed attention.")

    image |  patch |  tokens N |     attn ~ N^2
------------------------------------------------
   224px |   32px |        49 |          2,401
   224px |   16px |       196 |         38,416
   224px |    8px |       784 |        614,656
   384px |   16px |       576 |        331,776
   512px |   16px |     1,024 |      1,048,576

Halving the patch size 4x's the token count and ~16x's the attention cost.
Bigger images do the same -> high-res ViTs are costly; this is why Swin uses windowed attention.


In [5]:
# OPTIONAL: load a real pretrained ViT (downloads ~330MB of weights).
# Gated behind ALLOW_DOWNLOAD so the notebook runs offline; shows the call shape.
import importlib.util, os

if importlib.util.find_spec("torchvision") is not None and os.getenv("ALLOW_DOWNLOAD"):
    from torchvision.models import vit_b_16, ViT_B_16_Weights

    net = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1).eval()
    with torch.no_grad():
        pred = net(torch.randn(1, 3, 224, 224)).argmax(-1).item()
    print("vit_b_16 loaded -> predicted ImageNet class id:", pred)
    print("params:", f"{sum(p.numel() for p in net.parameters()):,}  (~86M, this is ViT-Base)")
else:
    print("Skipping pretrained ViT (set ALLOW_DOWNLOAD=1 and install torchvision to run it).")
    print("Shape: vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)(x), x=(B,3,224,224) -> (B,1000).")

Skipping pretrained ViT (set ALLOW_DOWNLOAD=1 and install torchvision to run it).
Shape: vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)(x), x=(B,3,224,224) -> (B,1000).


## 6. Gotchas & Pitfalls

- **Training from scratch on small data disappoints.** Without CNN priors, a from-scratch ViT on ImageNet-1k
  *loses* to a ResNet. Use a pretrained backbone, or the DeiT recipe (strong augmentation: RandAugment,
  Mixup/CutMix, repeated aug, + distillation). Don't benchmark a scratch-trained ViT and conclude "ViTs are
  bad."
- **Positional embeddings are tied to one resolution.** Change the input size (or patch grid) and you must
  **interpolate** the position embeddings (and usually fine-tune). Feeding a different resolution without
  interpolation silently breaks or crashes on a shape mismatch.
- **Image size must be divisible by patch size.** A leftover strip of pixels is dropped or errors out. 224/16
  = 14 works; 230/16 does not.
- **Attention cost is quadratic in tokens, not pixels.** Doubling resolution at fixed patch size quadruples N
  and ~16×'s attention FLOPs/memory. Don't naively crank resolution — shrink the patch *or* use a hierarchical
  model (Swin) instead.
- **Normalization must match the pretrained weights.** Pretrained ViTs expect specific mean/std normalization
  (and often a particular resize/crop). Feeding 0–1 or ImageNet-vs-CLIP-mismatched stats quietly tanks
  accuracy.
- **CLS vs mean-pooling is a real choice.** Many implementations average the patch tokens instead of using
  CLS; they perform similarly, but mixing conventions (train with CLS, infer with pooling) gives garbage.
- **ViTs can be less robust to small-data overfitting and need warmup.** They're sensitive to optimizer
  (AdamW), LR warmup, and weight decay; the LR schedule that works for ResNets often diverges a ViT.

## 7. When to Use vs Alternatives

| Option | Inductive bias | Cost vs resolution | Data needs | Best when |
|---|---|---|---|---|
| **ViT** | Minimal (global from layer 1) | O(N²) in patches — steep | High (pretrain / heavy aug) | Lots of data or good pretrained weights; global context; shared multimodal encoder |
| **CNN (ResNet/ConvNeXt)** | Strong locality + translation equivariance | O(pixels) — gentle | Modest; works from scratch | Small/medium data, tight latency, strong baseline with little tuning |
| **Swin / hierarchical ViT** | Local windows + hierarchy (CNN-like) | ~O(N) via windowed attention | Medium | High-res images, dense prediction (detection, segmentation) |
| **DeiT** | Same as ViT + distillation token | O(N²) | ImageNet-1k only (no JFT) | You want a ViT but can't do giant-scale pretraining |
| **Hybrid (CNN stem + Transformer)** | Conv early, attention late | Between the two | Medium | Want conv's data efficiency *and* global mixing |

**Rules of thumb.** Small dataset, no pretraining, need it working fast? Use a CNN. Plenty of data or a
pretrained ViT/DINOv2 backbone available? ViT will usually win and transfers beautifully. Dense prediction or
high resolution? Reach for a hierarchical/windowed ViT (Swin) to dodge the O(N²) wall. Building a multimodal
system (image + text)? ViT is the de-facto image encoder (CLIP, LLaVA-style models), so it's the natural
choice for representation sharing.

## 8. Resources

- **ViT paper** — Dosovitskiy et al., *An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale* (2020): https://arxiv.org/abs/2010.11929
- **Official code & checkpoints** — `google-research/vision_transformer`: https://github.com/google-research/vision_transformer
- **DeiT** — Touvron et al., *Training data-efficient image transformers & distillation through attention* (2021): https://arxiv.org/abs/2012.12877
- **Swin Transformer** — Liu et al., hierarchical windowed attention (2021): https://arxiv.org/abs/2103.14030
- **timm** — Ross Wightman's library, the practical home of pretrained ViTs and variants: https://github.com/huggingface/pytorch-image-models
- **The Annotated / Illustrated walkthrough** — HuggingFace ViT docs with diagrams and a runnable pipeline: https://huggingface.co/docs/transformers/model_doc/vit